# Acto 4 — Synthetic Fraud (Final Boss)

Combina las features de los actos previos sobre un dataset sintético con un **anillo de fraude plantado**.

## ¿Qué es detección de fraude en grafos?

Los grafos capturan *relaciones estructurales* entre entidades. En fraude financiero,
la señal no está en los valores individuales sino en los **patrones topológicos**:
- **Densidad inbound anómala:** una cuenta que recibe transferencias de muchas fuentes distintas.
- **Clusters aislados:** grupos de cuentas con alta cohesión interna y pocas conexiones externas.
- **Caminos cortos entre entidades sospechosas:** 2-3 saltos entre un sospechoso y fondos de origen.

## El dataset sintético

El generador crea 4 tipos de nodos:

| Tipo | Descripción | Rol en el fraude |
|------|-------------|------------------|
| `Person` (200) | Personas con nombre, país, descripción | 5 primeras = ring owners |
| `Account` (~312) | Cuentas bancarias con balance y moneda | 5 primeras = ring accounts |
| `LegalEntity` (~48) | Empresas legítimas con directores | Estructura corporativa |
| `ShellCompany` (~12) | Empresas fachada (20% de las companies) | Ocultar flujo de fondos |

Y 4 tipos de aristas: `OWNS`, `TRANSFERS`, `DIRECTS`, `CONTROLS`.

## El ring plantado

Las primeras 5 Persons y sus cuentas forman un **anillo cerrado**:
```
A1 ⟷ A2 ⟷ A3 ⟷ A4 ⟷ A5 ⟷ A1   (6 transfers por par + 2 cruzadas = ~14 inbound total)
```
Las cuentas no-ring reciben ≤4 transfers. Esa diferencia es la señal de detección.

Lee la narrativa completa en [`docs/tutorial/acto_4_synthetic_fraud/README.md`](../../docs/tutorial/acto_4_synthetic_fraud/README.md).

## 0. Setup

In [ ]:
%matplotlib inline
import sys, subprocess
from pathlib import Path

TUTORIALS_DIR = Path.cwd().resolve()
if TUTORIALS_DIR.name == "notebooks":
    TUTORIALS_DIR = TUTORIALS_DIR.parent
if str(TUTORIALS_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIALS_DIR))

import nopaldb
from shared import REPO_ROOT, db_path, load_nql
from shared.viz import execute_to_df
from shared.embeddings import attach_node_embeddings

DB = db_path("synthetic_fraud")
GENERATOR = TUTORIALS_DIR / "shared" / "synthetic_fraud_dataset.py"
print("DB target:", DB)

## 1. Generar DB si no existe

In [ ]:
if not Path(DB).exists():
    print(f"Generando {DB}...")
    subprocess.run(
        [sys.executable, str(GENERATOR), "--db", DB, "--reset"],
        cwd=str(REPO_ROOT),
        check=True,
    )
else:
    print(f"Reusando DB existente: {DB}")

graph = nopaldb.Graph.open(DB)
stats = graph.get_stats()
print(f"total_nodes: {int(stats['total_nodes'])}")
print(f"total_edges: {int(stats['total_edges'])}")

## 2. Paso 1 — Topología por label

El primer paso en cualquier investigación de grafo es **entender qué hay**: cuántos
nodos de cada tipo existen. Esto funciona como un *schema discovery* empírico —
NopalDB infiere el schema a partir de los labels observados.

Una distribución normal debería tener:
- Muchas Persons y Accounts (entidades principales)
- Menos Companies (estructuras de control)
- Balance coherente entre `LegalEntity` y `ShellCompany`

Si el conteo de un tipo fuera inesperadamente alto, eso por sí solo es una señal.

In [ ]:
df_topo = execute_to_df(graph, load_nql("acto_4", "01_topology.nql"))
df_topo

In [ ]:
# Visualización: distribución de nodos por tipo ─────────────────────────
import matplotlib.pyplot as plt

_label_colors = {
    "Person": "#3498db", "Account": "#27ae60",
    "LegalEntity": "#9b59b6", "ShellCompany": "#e74c3c",
}
_labels = df_topo['etiqueta'].tolist()
_totals = df_topo['total'].tolist()
_bar_colors = [_label_colors.get(l, '#95a5a6') for l in _labels]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(_labels, _totals, color=_bar_colors, edgecolor='white', width=0.6)
ax.set_ylabel('Cantidad de nodos')
ax.set_title('Distribución de nodos por tipo')
for bar, val in zip(bars, _totals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(val)), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(_totals) * 1.18)
plt.tight_layout()
plt.show()
print(f"Total nodos: {sum(_totals):.0f}")

## 3. Paso 2 — Detección del ring por densidad inbound

### ¿Por qué densidad inbound?

En un grafo de transferencias, la **densidad inbound** (cuántas cuentas distintas
te transfieren dinero) es una señal de anomalía.

- Una cuenta normal recibe dinero de 1-4 fuentes (salario, pagos ocasionales).
- Una cuenta en un **ring de fraude** recibe transfers de *todas las demás cuentas
  del ring* — muchas fuentes, monto similar, timestamp cercano.

Esta técnica se llama **degree-based anomaly detection** y es uno de los enfoques
más rápidos para fraud screening inicial (O(E) sobre las aristas).

El gate central del Acto 4: las **5 cuentas con más inbound son las del ring**
(cada una recibe 14, vs ≤4 para cuentas no-ring).

In [ ]:
df_inbound = execute_to_df(graph, load_nql("acto_4", "02_top_inbound.nql"))
df_inbound.head(10)


In [ ]:
# Identificamos las 5 ring accounts: las primeras 5 Persons, sus primeras Accounts.
ring_persons_q = execute_to_df(graph, '''
find p.id, p.name, a.id
from (p:Person) -[:OWNS]-> (a:Account)
limit 5
''')
ring_aids = ring_persons_q["a.id"].tolist()
print("Ring account IDs:")
for pid, pname, aid in zip(ring_persons_q["p.id"], ring_persons_q["p.name"], ring_aids):
    print(f"  {pname:<22} acct={aid[:8]}")

top5 = df_inbound.head(5)["b.id"].tolist()
matched = set(top5) & set(ring_aids)
print(f"\nTop-5 inbound coincide con el ring: {len(matched)}/5 cuentas")

In [ ]:
# Histograma: distribución de densidad inbound ───────────────────────────
# ¿Cuántas transferencias inbound recibe cada cuenta?
# El ring se delata como outlier en la cola derecha.
import matplotlib.pyplot as plt

_df_all_inbound = execute_to_df(graph, '''
find b.id, count(*) as inbound
from (a:Account) -[:TRANSFERS]-> (b:Account)
group by b.id
order by inbound desc
''')

_ring_set = set(ring_aids)
_inbound_vals = _df_all_inbound['inbound'].astype(int).tolist()
_acct_ids     = _df_all_inbound['b.id'].tolist()
_ring_inb  = [v for v, a in zip(_inbound_vals, _acct_ids) if a in _ring_set]
_norm_inb  = [v for v, a in zip(_inbound_vals, _acct_ids) if a not in _ring_set]

_max_val = max(_inbound_vals) + 1
_bins = range(0, _max_val + 2)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(_norm_inb, bins=_bins, color='#3498db', alpha=0.8,
        label=f'Cuentas normales (n={len(_norm_inb)})', edgecolor='white')
ax.hist(_ring_inb, bins=_bins, color='#e74c3c', alpha=0.95,
        label=f'Ring accounts (n={len(_ring_inb)})', edgecolor='white')
ax.axvline(x=14, color='#c0392b', linestyle='--', linewidth=2,
           label='Umbral ring: 14')
ax.set_xlabel('Transferencias inbound recibidas')
ax.set_ylabel('Número de cuentas')
ax.set_title('Distribución de densidad inbound — el ring es un outlier claro')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f'Ring inbound: {_ring_inb}  |  Normal max: {max(_norm_inb) if _norm_inb else 0}')

In [ ]:
# Visualización: topología del ring de fraude ────────────────────────────
# Muestra las 5 cuentas ring y sus transferencias internas.
# El layout circular subraya la naturaleza de anillo.
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

_all_tx = execute_to_df(graph, '''
find a.id as src, b.id as tgt
from (a:Account) -[:TRANSFERS]-> (b:Account)
''')
_rs = set(ring_aids)
_ring_tx = _all_tx[_all_tx['src'].isin(_rs) & _all_tx['tgt'].isin(_rs)]
_edge_w   = _ring_tx.groupby(['src','tgt']).size().reset_index(name='w')

_aid_to_name = dict(zip(ring_persons_q['a.id'], ring_persons_q['p.name']))
_short = {a: _aid_to_name.get(a, a[:6]).split()[0] for a in ring_aids}

_G_ring = nx.DiGraph()
for _a in ring_aids:
    _G_ring.add_node(_a)
for _, _row in _edge_w.iterrows():
    _G_ring.add_edge(_row['src'], _row['tgt'], weight=_row['w'])

_n = len(ring_aids)
_angles = np.linspace(0, 2*np.pi, _n, endpoint=False)
_pos_ring = {_a: (float(np.cos(_ang)), float(np.sin(_ang)))
             for _a, _ang in zip(ring_aids, _angles)}
_widths = [max(0.5, _G_ring.edges[e]['weight'] * 0.25) for e in _G_ring.edges()]

fig3, ax3 = plt.subplots(figsize=(8, 7))
nx.draw_networkx_nodes(_G_ring, _pos_ring, node_color='#e74c3c',
                       node_size=2000, ax=ax3, alpha=0.9)
nx.draw_networkx_labels(_G_ring, _pos_ring, labels=_short,
                        font_size=9, font_weight='bold', ax=ax3)
nx.draw_networkx_edges(_G_ring, _pos_ring, ax=ax3, arrows=True,
                       arrowstyle='-|>', arrowsize=12, edge_color='#888',
                       width=_widths, connectionstyle='arc3,rad=0.1')
ax3.set_title(
    f'Ring de fraude: {_n} cuentas · {len(_ring_tx)} transferencias internas\n'
    '(grosor de arista ∝ número de transfers paralelos)',
    fontsize=11, pad=10)
ax3.set_axis_off()
plt.tight_layout()
plt.show()
print(f'Transferencias intra-ring: {len(_ring_tx)}  |  Promedio inbound/cuenta: {len(_ring_tx)/_n:.1f}')

## 4. Paso 3 — Community detection (Leiden) sobre las cuentas

### ¿Qué es community detection?

Un **algoritmo de detección de comunidades** particiona los nodos de un grafo en
grupos con alta cohesión interna y poca conectividad entre grupos. Matemáticamente
maximiza la **modularidad Q**:

```
Q = Σ [ (aristas internas al grupo) / (total aristas) - (grado²) / (total grados²) ]
```

### Louvain vs Leiden

| Algoritmo | Garantía | Complejidad | Uso |
|-----------|----------|-------------|-----|
| **Louvain** | Ninguna (puede producir comunidades inconexas) | O(E log V) | Exploración rápida |
| **Leiden** | Comunidades bien conectadas (Traag et al. 2019) | O(E log V) | Producción |

NopalDB usa `leiden(n)` por defecto en NQL. Leiden no puede producir
una comunidad donde los nodos no estén conectados entre sí — esto lo hace
más confiable para fraud screening.

### Limitación esperada

El ring **no necesariamente colapsa en UNA comunidad**. Cada ring account tiene su
propio owner (`Person`), lo que genera conexiones hacia fuera del ring. Leiden
particiona finamente con grafos heterogéneos. Pero las cuentas del ring tienden
a quedar en comunidades de **mayor tamaño** que las no-ring.

In [ ]:
df_comm = execute_to_df(graph, load_nql("acto_4", "04_communities.nql"))
ring_communities = df_comm[df_comm["a.id"].isin(ring_aids)]["community"].tolist()
print(f"Comunidades de las cuentas ring: {ring_communities}")
print(f"Comunidades unicas: {sorted(set(ring_communities))}")

In [ ]:
# Visualización: comunidades detectadas (subgrafo de cuentas) ────────────
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from collections import Counter

# Subgrafo de transferencias entre cuentas
_df_tx_vis = execute_to_df(graph, '''
find a.id as src, b.id as tgt
from (a:Account) -[:TRANSFERS]-> (b:Account)
limit 500
''')

# Mapas auxiliares
_aid_to_comm = dict(zip(df_comm['a.id'], df_comm['community'].astype(int)))
_aid_to_name = dict(zip(ring_persons_q['a.id'], ring_persons_q['p.name']))
_ring_s = set(ring_aids)

# Top 6 comunidades mas frecuentes
_comm_counts = Counter(_aid_to_comm.values())
_top_comms   = [c for c, _ in _comm_counts.most_common(6)]
_cmap_fn     = plt.get_cmap('Set2', len(_top_comms))
_comm_color  = {c: _cmap_fn(i) for i, c in enumerate(_top_comms)}

_G_c = nx.DiGraph()
for _, _row in _df_tx_vis.iterrows():
    _G_c.add_edge(_row['src'], _row['tgt'])

_acc_nodes  = list(_G_c.nodes())
_node_cols  = []
_node_sizes = []
for _nd in _acc_nodes:
    _is_ring = _nd in _ring_s
    _c = _aid_to_comm.get(_nd, -1)
    _node_cols.append('#e74c3c' if _is_ring else _comm_color.get(_c, '#cccccc'))
    _node_sizes.append(500 if _is_ring else 150)

fig_c, ax_c = plt.subplots(figsize=(11, 8))
_pos_c = nx.spring_layout(_G_c, seed=42, k=0.55)
nx.draw_networkx_edges(_G_c, _pos_c, ax=ax_c, alpha=0.12, edge_color='#aaa', arrows=False)
nx.draw_networkx_nodes(_G_c, _pos_c, nodelist=_acc_nodes,
                       node_color=_node_cols, node_size=_node_sizes, ax=ax_c, alpha=0.88)
# Etiquetar solo ring accounts visibles en el subgrafo
_ring_vis = [a for a in ring_aids if a in _G_c.nodes()]
_rlabels  = {a: _aid_to_name.get(a, a[:6]).split()[0] for a in _ring_vis}
nx.draw_networkx_labels(_G_c, _pos_c, labels=_rlabels,
                        font_size=7, font_weight='bold', ax=ax_c)
ax_c.set_title(
    'Community detection (Leiden) — rojo=ring accounts plantados, colores=comunidades detectadas',
    fontsize=10)
ax_c.set_axis_off()
ax_c.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Ring accounts (fraud)'),
    mpatches.Patch(color='#cccccc', label='Otras cuentas (comunidad varía)'),
], loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()
print(f'Comunidades de los ring accounts: {ring_communities}')
print('Leiden los separa porque cada ring account tiene también conexión con su Person owner.')

## 5. Paso 4 — Embeddings semánticos sobre descripciones de Person

### ¿Qué son los embeddings y por qué usarlos en fraude?

Un **embedding** es una representación vectorial densa (128-768 dimensiones) de
texto, nodo o grafo. Modelos como **MiniLM** (22M parámetros) convierten un
texto en un punto en ℝⁿ donde la distancia coseno captura similitud semántica.

En detección de fraude, los embeddings permiten:
- **Clustering semántico:** personas con descripciones similares (misma actividad,
  mismo país) aparecen cercanas en el espacio vectorial.
- **Anomaly detection:** entidades con descripción atípica quedan lejos del centro.
- **Feature extraction para GNN:** los embeddings se pasan como features de nodo
  a un Graph Neural Network.

### Pipeline en NopalDB

```
Texto (description) → MiniLM → vector[384] → almacenado en nodo
                                            ↓
                              HNSW index (búsqueda ANN en O(log N))
                                            ↓
                              similar_to(n, 'Alice Novak', 'minilm')
```

Cada Person tiene una descripción textual generada determinísticamente.
La señal es débil (descripciones genéricas) pero el pipeline es válido.

In [ ]:
items = []
res = graph.execute_nql("find p.id, p.description from (p:Person)")
for r in getattr(res, "query", res):
    items.append((r["p.id"], r["p.description"]))
print(f"Persons con descripcion: {len(items)}")
n = attach_node_embeddings(
    graph,
    items,
    cache_name="acto4_fraud_persons_minilm",
    model_label="minilm",
)
print(f"embeddings adjuntados: {n}")

In [ ]:
# Tomamos la primera Person del ring y buscamos sus 5 vecinas semanticas
ring_person_name = ring_persons_q.iloc[0]["p.name"]
import pandas as pd
if n > 0:
    df_sim = execute_to_df(graph, f"""
find p.name, p.country
from (p:Person)
where similar_to(p, "{ring_person_name}", "minilm")
limit 5
""")
else:
    df_sim = pd.DataFrame(columns=["p.name", "p.country"])
    print("Nota v0.4.19: similar_to omitido porque el binding Python no adjunto embeddings.")
df_sim


## 6. Paso 5 — Arrow export para ML

### ¿Por qué Apache Arrow?

**Apache Arrow** es un formato columnar en memoria diseñado para transferencia
de datos de alta velocidad entre sistemas. Su característica clave: **zero-copy**.

```
NopalDB (Rust)  →  Arrow IPC bytes  →  pandas DataFrame
                                     →  PyTorch Geometric (GNN)
                                     →  scikit-learn (ML clásico)
                                     →  Spark / DuckDB
```

Sin Arrow, cada transferencia requeriría serialización/deserialización costosa.
Arrow usa el mismo layout de memoria en Rust y Python — el `memcpy` es
literalmente cero bytes de conversión.

`graph.to_arrow()` produce un `RecordBatch` columnar de los nodos,
listo para pandas/PyG/sklearn. El resultado incluye `id`, `label` y
`property_count` por nodo.

In [ ]:
import pyarrow as pa
import pyarrow.ipc as ipc
import io

# graph.to_arrow() devuelve bytes en formato IPC stream. Lo deserializamos.
arrow_bytes = graph.to_arrow()
print(f"raw bytes: {len(arrow_bytes)} bytes")
reader = ipc.open_stream(io.BytesIO(arrow_bytes))
table = reader.read_all()
print(f"Arrow Table: rows={table.num_rows} cols={table.num_columns}")
print(f"columns: {table.column_names[:8]}{'...' if len(table.column_names) > 8 else ''}")
df_arrow = table.to_pandas()
df_arrow.head()

## 7. Gate de verificación cruzada

Invariantes del Acto 4:
- Top-5 cuentas inbound = los 5 ring accounts (14 transfers cada una).
- Conteo correcto por label.

In [ ]:
# Gate cruzado: el ring se delata por densidad inbound.
# - Exactamente 5 cuentas tienen >= 14 transfers inbound.
# - Las demas tienen < 14.
# - Topologia esperada: 200 Persons + 312 Accounts.
high_inbound = df_inbound[df_inbound["inbound"] >= 14]
print(f"Cuentas con inbound >= 14: {len(high_inbound)}")
print(high_inbound)
assert len(high_inbound) == 5, f"Esperaba 5 cuentas con 14+ inbound, obtuve {len(high_inbound)}"
assert all(n == 14 for n in high_inbound["inbound"]), f"Counts: {high_inbound['inbound'].tolist()}"

topo_dict = dict(zip(df_topo["etiqueta"], df_topo["total"]))
assert topo_dict.get("Person") == 200
assert topo_dict.get("Account") == 312

print("\nGATE OK — el ring fraudulento se delata por densidad inbound (5 cuentas con 14 transfers cada una).")

## Cierre

Has recorrido los 4 actos del tutorial NopalDB:
1. **Florentine** — algoritmos clásicos de centralidad y community.
2. **Synthetic Offshore Network** — embeddings + HNSW + path queries con anomalía.
3. **Biomedical OWL** — reasoner CR1+CR2+CR3.
4. **Synthetic Fraud** — combinación final con detección estructural.

Próximos pasos sugeridos: aplicar los patrones a un dataset propio, escribir queries NQL con tu schema, integrar Arrow export con un pipeline de ML.